# GridSight UK — Train trên Colab (GPU)

Stack **TCN-Q + LGBM-Q → Linear-Q** (quantile q10/q50/q90).
Chạy lần lượt từng cell. **Không cần GitHub** — chỉ upload `modeling.zip` + tải Gold từ HF.

> Trước khi bắt đầu: **Runtime → Change runtime type → T4 GPU → Save**.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa bật GPU! Runtime -> Change runtime type -> T4 GPU'

## 2. Đưa code lên Colab (upload `modeling.zip`)

Ở **máy local**, tạo zip của thư mục `modeling/`:
```bash
cd /Users/hoangdat/Data/Projects/test_gridsight-uk
zip -r modeling.zip modeling
```
Rồi chạy cell dưới và chọn file `modeling.zip`.

In [ ]:
from google.colab import files
up = files.upload()                 # chọn modeling.zip
!unzip -oq modeling.zip -d .
!ls modeling | head
print('OK: modeling/ đã sẵn sàng')

## 3. Cài thư viện
Colab đã có **torch (GPU)** — KHÔNG cài lại torch, chỉ thêm vài gói nhẹ.

In [ ]:
!pip install -q lightgbm scikit-learn joblib huggingface_hub loguru matplotlib

## 4. Tải Gold từ HF (đã có sẵn 25 parquet)
Dán **HF token** (có quyền đọc repo team) khi được hỏi.

In [ ]:
from getpass import getpass
from huggingface_hub import snapshot_download
tok = getpass('HF token: ')
snapshot_download('gridsight-team/gridsight-gold', repo_type='dataset',
                  local_dir='data/gold', token=tok)
!echo 'Gold files:' && find data/gold -name '*.parquet' | wc -l

## 5. Smoke test (nhanh ~1 phút) — kiểm tra chạy được
Bỏ qua được nếu muốn train thẳng.

In [ ]:
!python -m modeling --fast

## 6. TRAIN thật (GPU tự dùng)
Tuỳ chọn: `--target target_cf|target_mw`, `--val-start`, `--test-start`, `--seq-len`, `--n-folds`.
Log in tiến độ từng OOF fold + từng epoch của TCN, rồi `[val]` / `[test]`.

In [ ]:
!python -m modeling --epochs 40

## 7. Biểu đồ đánh giá mô hình
Fan chart (q10–q90 vs actual) + dashboard (calibration, scatter q50, MAE theo giờ, thẻ metric).

In [ ]:
!python -m modeling.evaluate --split test
from IPython.display import Image, display
display(Image('artifacts/model/plots/evaluation_fan.png'))
display(Image('artifacts/model/plots/evaluation_dashboard.png'))

## 8. Tải model + biểu đồ về

In [ ]:
import json
print(json.dumps(json.load(open('artifacts/model/metrics.json')), indent=2))
!zip -rq artifacts.zip artifacts/model
from google.colab import files
files.download('artifacts.zip')   # stack.joblib + tcn.pt + metrics.json + plots/

### (Tuỳ chọn) Lưu vào Google Drive để khỏi mất khi hết session

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r artifacts/model /content/drive/MyDrive/gridsight_artifacts

---
**Chia dữ liệu (theo thời gian, không xáo trộn):** Train `< 2024-07-01` (~2023 + H1-2024) · Val `2024-07..09` · Test `2024-10..12`. OOF 5-fold nằm trong Train để huấn luyện meta-learner. Đổi mốc bằng `--val-start` / `--test-start`.

**Metrics:** pinball (càng thấp càng tốt), coverage q10–q90 (≈ 80%), crossing_rate (= 0), `skill_vs_neso_q50` (> 0 = đánh bại baseline NESO).